# 07 — QSP / QSVT: Entry Point

**Trilha:** Ponte para a fronteira  
**Milestone:** 4 — Ponte honesta para a fronteira

---

## Pergunta

O que QSP/QSVT realmente estão fazendo? Por que eles atuam sobre transformações polinomiais? E onde valores singulares bastam versus onde autovalores passam a importar?

Este notebook não tenta provar nada novo. O objetivo é chegar perto do vocabulário correto para poder ler papers da área com criticidade.

## Modelo matemático mínimo

### Quantum Signal Processing (QSP)

**Ideia central:** dado um sinal $x \in [-1,1]$, construir transformação quântica que aplica um polinômio $P(x)$ ao sinal.

Para 1 qubit com sinal $x = \cos\theta$, o operador base é:
$$W(x) = \begin{pmatrix} x & i\sqrt{1-x^2} \\ i\sqrt{1-x^2} & x \end{pmatrix}$$

Intercalando $W(x)$ com rotações de fase $e^{i\phi_k Z}$:
$$U_{QSP} = e^{i\phi_0 Z} \prod_{k=1}^d W(x) e^{i\phi_k Z}$$

O bloco $(0,0)$ deste operador é um polinômio $P(x)$ de grau $d$.

**Resultado central:** qualquer polinômio $P$ com paridade definida e $|P(x)| \leq 1$ pode ser realizado com alguma escolha de fases $\{\phi_k\}$.

### Quantum Singular Value Transformation (QSVT)

QSVT generaliza QSP para matrizes. Se $A$ tem block-encoding $U_A$, QSVT aplica $P(\sigma_i)$ a cada valor singular $\sigma_i$ de $A$:
$$A = U\Sigma V^\dagger \xrightarrow{QSVT} U P(\Sigma) V^\dagger$$

Exemplos de $P$ e o algoritmo resultante:
- $P(x) = x$ → identidade (trivial)
- $P(x) \approx 1/x$ → inversão de matriz (HHL)
- $P(x) \approx \text{sign}(x)$ → amplificação de amplitude
- $P(x) = e^{itx}$ → Hamiltonian simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# --- QSP 1D: verificação do mecanismo polinomial ---

def W(x):
    """Operador sinal de QSP para x em [-1,1]."""
    s = np.sqrt(1 - x**2 + 0j)
    return np.array([[x, 1j*s], [1j*s, x]], dtype=complex)

def Rz(phi):
    """Rotação em Z."""
    return np.array([[np.exp(1j*phi), 0], [0, np.exp(-1j*phi)]], dtype=complex)

def qsp_circuit(x, phases):
    """Circuito QSP com fases dadas, avaliado em x."""
    U = Rz(phases[0])
    for phi in phases[1:]:
        U = U @ W(x) @ Rz(phi)
    return U

# Polinômio alvo: P(x) = x (grau 1)
# Com fases (0, 0), o bloco (0,0) de W(x) é exatamente x
x_vals = np.linspace(-1, 1, 100)
phases_identity = [0.0, 0.0]  # trivial

P_computed = [qsp_circuit(x, phases_identity)[0, 0] for x in x_vals]
P_computed = np.array(P_computed)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x_vals, x_vals, 'k--', label='P(x) = x (target)', linewidth=2)
axes[0].plot(x_vals, P_computed.real, 'b-', label='Re[U(0,0)] = QSP', alpha=0.7)
axes[0].set_title('QSP: P(x) = x (grau 1, fases triviais)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('P(x)')
axes[0].legend()

# Polinômio de grau 3: P(x) = x^3 (odd, |P| <= 1 em [-1,1])
# Fases para x^3 (calculadas analiticamente)
phi_cube = [np.pi/2, -np.pi/2, np.pi/2, -np.pi/2]  # aproximação
P_cube = [qsp_circuit(x, phi_cube)[0, 0] for x in x_vals]
P_cube = np.array(P_cube)

axes[1].plot(x_vals, x_vals**3, 'k--', label='P(x) = x³ (target)', linewidth=2)
axes[1].plot(x_vals, P_cube.real, 'r-', label='Re[U(0,0)] com fases', alpha=0.7)
axes[1].set_title('QSP: tentativa de P(x) = x³')
axes[1].set_xlabel('x'); axes[1].set_ylabel('P(x)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('QSP: intercalar W(x) com rotações de fase realiza um polinômio no bloco (0,0).')
print('Encontrar as fases corretas para um P(x) dado é o problema de "phase finding".')

In [ ]:
# --- QSVT: aplicando polinômio aos valores singulares via block-encoding ---

def block_encode(A):
    """Block-encoding simples de A (||A|| <= 1)."""
    n = A.shape[0]
    I = np.eye(n, dtype=complex)
    
    def mat_sqrt(M):
        vals, vecs = np.linalg.eigh(M)
        return vecs @ np.diag(np.sqrt(np.maximum(vals, 0))) @ vecs.conj().T
    
    B = mat_sqrt(I - A @ A.conj().T)
    C = mat_sqrt(I - A.conj().T @ A)
    return np.block([[A, B], [C, -A.conj().T]])

# A com valores singulares conhecidos: σ₁=0.8, σ₂=0.3
U_svd = np.array([[0.6, 0.8], [-0.8, 0.6]])  # unitária aleatória
V_svd = np.array([[0.7, 0.714], [-0.714, 0.7]])  # outra unitária
Sigma = np.diag([0.8, 0.3])
A_target = U_svd @ Sigma @ V_svd.T

_, sigma_A, _ = np.linalg.svd(A_target)
print(f'A construída com σ = {sigma_A.round(4)}')

# Block-encoding
U_A = block_encode(A_target)
print(f'Block-encoding U_A é unitário: {np.allclose(U_A @ U_A.conj().T, np.eye(4))}')

# Recuperar A do bloco
A_recovered = U_A[:2, :2]
_, sigma_recovered, _ = np.linalg.svd(A_recovered)
print(f'σ recuperados do bloco: {sigma_recovered.round(4)}')
print()

# QSVT conceitualmente: aplicar P(σ) aos valores singulares
# Para P(x) = x^2: A -> U Σ² V†
P_sigma = sigma_recovered**2  # P(x) = x²
A_transformed_expected = U_svd @ np.diag(P_sigma) @ V_svd.T
_, sigma_transformed, _ = np.linalg.svd(A_transformed_expected)

print('Transformação QSVT com P(x) = x²:')
print(f'  σ antes:  {sigma_A.round(4)}')
print(f'  σ depois: {sigma_transformed.round(4)}')
print(f'  σ² esperados: {sigma_A**2}')

In [ ]:
# --- Por que eigenvalue processing (além de QSVT) ---

# QSVT processa valores singulares: A -> P(|eigenvalues of Hermitian part|)
# Para Hamiltoniano H Hermitiano: autovalores são reais, valores singulares = |autovalores|
# Mas: sinal do autovalor se perde! E₋₁ = -1 e E₊₁ = +1 -> σ₁ = σ₂ = 1

# Exemplo: distinção de energia
H = np.array([[1, 0.5], [0.5, -1]], dtype=complex)  # autovalores: ≈ +1.12 e ≈ -1.12
eigvals_H = np.linalg.eigvalsh(H)
_, sigma_H, _ = np.linalg.svd(H)

print('Por que eigenvalue processing vai além do QSVT:')
print('='*55)
print(f'H = [[1, 0.5], [0.5, -1]]')
print(f'Autovalores de H: {eigvals_H.round(4)}')
print(f'Valores singulares de H: {sigma_H.round(4)}')
print()
print('QSVT pode aplicar P(σ) mas perde o SINAL dos autovalores.')
print('Para ground state preparation, o sinal importa:')
print('  E_ground = {:.4f}  (não é apenas |E|)'.format(eigvals_H[0]))
print()
print('Eigenvalue processing (Dong et al.) trabalha diretamente com')
print('os autovalores do Hamiltoniano, preservando o sinal.')
print('Isso é necessário para: ground state preparation, real-time dynamics,')
print('e estimativas de energia em early fault-tolerant hardware.')

# Visualização: autovalores reais preservam sinal
fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(-2, 2, 200)

# P(x) = 1/(x - E_0) : diverge na energia do ground state
# QSVT usaria P(σ): não vê o sinal
E0 = eigvals_H[0]
ax.plot(x, np.abs(x), 'b-', label='Valores singulares |x|')
ax.plot(x, x, 'r-', label='Autovalores x (preserva sinal)')
ax.axvline(E0, color='green', linestyle='--', label=f'E_ground = {E0:.2f}')
ax.axvline(-E0, color='gray', linestyle=':', alpha=0.5, label=f'-E_ground (QSVT não distingue)')
ax.set_xlabel('Valor espectral')
ax.set_title('Sinal importa: eigenvalue vs. singular value processing')
ax.legend(fontsize=9)
ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

## Conclusão

1. **QSP é uma linguagem para descrever transformações polinomiais de sinais.** Intercalar o operador sinal $W(x)$ com rotações de fase realiza um polinômio $P(x)$ no bloco de saída. Grau $d$ → $d$ aplicações de $W$.

2. **QSVT estende QSP para matrizes via block-encoding.** Em vez de um escalar $x$, o "sinal" é uma matriz $A$. QSP é aplicado a cada valor singular independentemente.

3. **QSVT unifica algoritmos quânticos:** Hamiltonian simulation, busca, inversão de matriz, amplitude amplification — todos são escolhas diferentes de polinômio $P$.

4. **O limite do QSVT:** processa valores singulares $\sigma_i \geq 0$ — perde o sinal dos autovalores de Hamiltonianos Hermitianos. Para ground state preparation e estimativa de energia, $E_{-}$ e $E_{+}$ têm o mesmo valor singular mas autovalores distintos.

5. **Eigenvalue processing (Dong et al. 2022)** resolve esse limite, trabalhando diretamente com os autovalores reais. É o que permite preparação de ground state em hardware early fault-tolerant.

6. **O que este notebook não faz:** não deriva as fases de QSP, não implementa QSVT completo, não prova o teorema de completude polinomial. Esses são os próximos passos para quem quer ler os papers originais.

---

**Fim da Milestone 4.** Onde continuar: Low & Chuang (2017), Gilyén et al. (2019), Dong et al. (2022). Ver [references/papers.md](../references/papers.md).